# 🍇 GRAPE — DATA COLLECTION & VERIFICATION AUDIT
### Project: Plant Disease Detection (Computer Vision)
**Internal Source:** PlantVillage Laboratory Dataset (Controlled lab background)  
**External Source:** Official GVLiD Dataset via Mendeley Data (DOI: `10.17632/wkymf8bhcg.5` — Field vineyard images from Maharashtra, India)  
**Classes (3):** `Healthy`, `Black_Rot`, `Leaf_Blight` (Isariopsis)  
**Target:** 300 Images per class (Internal: 300/class, External: 300/class → Total: 1,800 images balanced)



In [2]:
# ================================================================
# 🍇 GRAPE — STEP 1: VERIFY INTERNAL + EXTRACT & DEDUPLICATE GVLiD
# ================================================================

import os
import shutil
import hashlib
import random
import glob
import subprocess
from pathlib import Path
from PIL import Image

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

PROJECT = '/content/drive/MyDrive/Plant Disease Detection (Computer Vision)'
if not os.path.exists(PROJECT):
    PROJECT = r'G:\My Drive\Plant Disease Detection (Computer Vision)'

GRAPE_ROOT = os.path.join(PROJECT, '1st Raw_Data', 'Grape')
INTERNAL_DIR = os.path.join(GRAPE_ROOT, 'Internal_PlantVillage')
EXTERNAL_DIR = os.path.join(GRAPE_ROOT, 'External_Natural')
FINAL_DIR = os.path.join(GRAPE_ROOT, 'Final_Combined')
TEST_REALWORLD_DIR = os.path.join(PROJECT, '7th Test_Images', 'Grape_RealWorld')

CLASSES = ['Healthy', 'Black_Rot', 'Leaf_Blight']
TARGET_PER_CLASS = 300
TEST_PER_CLASS = 10
SEED = 42
random.seed(SEED)

for d in [INTERNAL_DIR, EXTERNAL_DIR, FINAL_DIR]:
    for c in CLASSES:
        os.makedirs(os.path.join(d, c), exist_ok=True)

for c in CLASSES:
    os.makedirs(os.path.join(TEST_REALWORLD_DIR, c), exist_ok=True)

print('=' * 80)
print('🍇 GRAPE DATA COLLECTION & SETUP (GVLiD FIELD DATASET + DEDUPLICATION)')
print('=' * 80)

# ----------------------------------------------------------------
# PART 1: VERIFY INTERNAL BENCHMARK SAMPLES
# ----------------------------------------------------------------
print('\n1. Verifying Internal PlantVillage samples in Drive...')
for c in CLASSES:
    fld = os.path.join(INTERNAL_DIR, c)
    cur_count = len([f for f in os.listdir(fld) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]) if os.path.exists(fld) else 0
    print(f'  Internal {c:<15}: {cur_count} images found')

# ----------------------------------------------------------------
# PART 2: CHECK / EXTRACT GVLiD ARCHIVE
# ----------------------------------------------------------------
EXTRACT_DIR = '/content/gvlid_extracted'
RAR_PATH = '/content/GVLiD.rar'
GVLID_URL = 'https://data.mendeley.com/public-files/datasets/wkymf8bhcg/files/7aa4663b-c462-4283-b280-2f3df0be9472/file_downloaded'

if not os.path.exists(os.path.join(EXTRACT_DIR, 'DATASET')):
    print('\n2. Ensuring GVLiD archive is extracted...')
    if not os.path.exists(RAR_PATH) or os.path.getsize(RAR_PATH) < 800 * 1024 * 1024:
        print('  Downloading GVLiD.rar...')
        cmd = f'wget -c --show-progress -q "{GVLID_URL}" -O "{RAR_PATH}"'
        os.system(cmd)

    subprocess.run(['apt-get', 'install', '-y', 'unar', 'p7zip-full'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    if shutil.which('unar'):
        os.system(f'unar -q -o "{EXTRACT_DIR}" "{RAR_PATH}"')
    elif shutil.which('7z'):
        os.system(f'7z x -y -o"{EXTRACT_DIR}" "{RAR_PATH}"')
else:
    print('\n2. GVLiD dataset already extracted at:', EXTRACT_DIR)

# ----------------------------------------------------------------
# PART 3: DISCOVER FOLDERS
# ----------------------------------------------------------------
print('\n3. Discovering extracted field image directories...')
class_dirs = {}
for root, dirs, files in os.walk(EXTRACT_DIR):
    folder_name = os.path.basename(root).lower().strip()
    img_files = [f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if len(img_files) == 0:
        continue

    if 'healthy' in folder_name:
        class_dirs['Healthy'] = root
    elif 'black' in folder_name and 'rot' in folder_name:
        class_dirs['Black_Rot'] = root
    elif 'blight' in folder_name:
        class_dirs['Leaf_Blight'] = root

for c in CLASSES:
    found_dir = class_dirs.get(c, 'NOT FOUND')
    count = len([f for f in os.listdir(found_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]) if found_dir != 'NOT FOUND' else 0
    print(f'  Class {c:<15}: {count} candidate field images in {found_dir}')

# ----------------------------------------------------------------
# PART 4: STRICT HASH DEDUPLICATION & SELECTION (300 EXT + 10 TEST)
# ----------------------------------------------------------------
print('\n4. Deduplicating & selecting exactly 300 unique External + 10 Test images...')

# Clean out old external folders to remove previous duplicates
for c in CLASSES:
    dst_ext = os.path.join(EXTERNAL_DIR, c)
    dst_tst = os.path.join(TEST_REALWORLD_DIR, c)
    for folder in [dst_ext, dst_tst]:
        if os.path.exists(folder):
            for f in os.listdir(folder):
                try:
                    os.remove(os.path.join(folder, f))
                except Exception:
                    pass

total_external_saved = 0

for c in CLASSES:
    src_dir = class_dirs.get(c)
    if not src_dir or not os.path.exists(src_dir):
        print(f'  ❌ Directory for {c} not found!')
        continue

    all_files = [os.path.join(src_dir, f) for f in os.listdir(src_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    rng = random.Random(SEED)
    rng.shuffle(all_files)

    # Collect strictly unique images by MD5 hash
    seen_hashes = set()
    unique_images = []

    for fpath in all_files:
        try:
            with open(fpath, 'rb') as fb:
                h = hashlib.md5(fb.read()).hexdigest()
            if h in seen_hashes:
                continue

            with Image.open(fpath) as img:
                img.verify()

            seen_hashes.add(h)
            unique_images.append(fpath)
        except Exception:
            continue

    print(f'  {c:<15}: Found {len(unique_images)} strictly unique field images from {len(all_files)} files.')

    # Prepare 300 for External_Natural + 10 for Test_Images
    needed = TARGET_PER_CLASS + TEST_PER_CLASS  # 310
    final_pool = []

    for p in unique_images:
        final_pool.append(('original', p, 0))

    if len(final_pool) < needed:
        shortfall = needed - len(final_pool)
        print(f'    -> Augmenting {shortfall} unique field samples via geometric rotation/flip...')
        aug_idx = 0
        transforms = [Image.ROTATE_90, Image.ROTATE_180, Image.ROTATE_270, Image.FLIP_LEFT_RIGHT]
        while len(final_pool) < needed:
            base_img_path = unique_images[aug_idx % len(unique_images)]
            trans = transforms[(aug_idx // len(unique_images)) % len(transforms)]
            final_pool.append(('transformed', base_img_path, trans))
            aug_idx += 1

    # Save exactly 300 to External_Natural
    dst_ext = os.path.join(EXTERNAL_DIR, c)
    saved_ext = 0
    for idx in range(TARGET_PER_CLASS):
        item_type, src_path, trans = final_pool[idx]
        dst_file = os.path.join(dst_ext, f'Grape_External_{c}_{idx+1:04d}.jpg')
        if item_type == 'original':
            shutil.copy2(src_path, dst_file)
        else:
            with Image.open(src_path) as img:
                timg = img.transpose(trans)
                timg.save(dst_file, 'JPEG', quality=95)
        saved_ext += 1

    # Save exactly 10 to 7th Test_Images/Grape_RealWorld
    dst_tst = os.path.join(TEST_REALWORLD_DIR, c)
    saved_tst = 0
    for idx in range(TARGET_PER_CLASS, TARGET_PER_CLASS + TEST_PER_CLASS):
        item_type, src_path, trans = final_pool[idx]
        dst_file = os.path.join(dst_tst, f'Grape_Real_{c}_{saved_tst+1:02d}.jpg')
        if item_type == 'original':
            shutil.copy2(src_path, dst_file)
        else:
            with Image.open(src_path) as img:
                timg = img.transpose(trans)
                timg.save(dst_file, 'JPEG', quality=95)
        saved_tst += 1

    total_external_saved += saved_ext
    print(f'  ✅ External {c:<15}: 300 unique saved to External_Natural, 10 saved to Test_Images.')

print('\n' + '=' * 80)
print(f'✅ STEP 1 COMPLETE: Exactly {total_external_saved}/900 100% unique External field images saved!')
print('=' * 80)



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🍇 GRAPE DATA COLLECTION & SETUP (GVLiD FIELD DATASET + DEDUPLICATION)

1. Verifying Internal PlantVillage samples in Drive...
  Internal Healthy        : 300 images found
  Internal Black_Rot      : 300 images found
  Internal Leaf_Blight    : 300 images found

2. GVLiD dataset already extracted at: /content/gvlid_extracted

3. Discovering extracted field image directories...
  Class Healthy        : 1109 candidate field images in /content/gvlid_extracted/DATASET/healthy
  Class Black_Rot      : 808 candidate field images in /content/gvlid_extracted/DATASET/Black rot
  Class Leaf_Blight    : 672 candidate field images in /content/gvlid_extracted/DATASET/leaf blight

4. Deduplicating & selecting exactly 300 unique External + 10 Test images...
  Healthy        : Found 986 strictly unique field images from 1109 files.
  ✅ External Healthy        : 300 unique sav

In [3]:
# ================================================================
# 🍇 GRAPE — STEP 2: DATA INTEGRITY, DEDUPLICATION & FINAL COMBINED
# ================================================================

import os
import csv
import hashlib
import shutil
import pandas as pd
from PIL import Image

PROJECT = '/content/drive/MyDrive/Plant Disease Detection (Computer Vision)'
if not os.path.exists(PROJECT):
    PROJECT = r'G:\My Drive\Plant Disease Detection (Computer Vision)'

GRAPE = os.path.join(PROJECT, '1st Raw_Data', 'Grape')
INTERNAL = os.path.join(GRAPE, 'Internal_PlantVillage')
EXTERNAL = os.path.join(GRAPE, 'External_Natural')
FINAL = os.path.join(GRAPE, 'Final_Combined')
VERIF_DIR = os.path.join(GRAPE, 'Final_Verification')
os.makedirs(VERIF_DIR, exist_ok=True)

CLASSES = ['Healthy', 'Black_Rot', 'Leaf_Blight']

for c in CLASSES:
    os.makedirs(os.path.join(FINAL, c), exist_ok=True)

print('=' * 80)
print('🍇 GRAPE DATASET VERIFICATION & INTEGRITY AUDIT')
print('=' * 80)

summary_data = []
detailed_records = []
all_hashes = {}

# 1. Audit Internal & External
for src_name, src_dir in [('INTERNAL', INTERNAL), ('EXTERNAL', EXTERNAL)]:
    for c in CLASSES:
        folder = os.path.join(src_dir, c)
        if not os.path.exists(folder):
            continue
        files = sorted([f for f in os.listdir(folder) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        valid_count = 0
        corrupt_count = 0
        seen_hashes = set()
        dup_count = 0

        for f in files:
            fpath = os.path.join(folder, f)
            try:
                with Image.open(fpath) as img:
                    img.verify()
                with open(fpath, 'rb') as fb:
                    h = hashlib.md5(fb.read()).hexdigest()

                if h in seen_hashes:
                    dup_count += 1
                    is_dup = True
                else:
                    seen_hashes.add(h)
                    valid_count += 1
                    is_dup = False

                all_hashes.setdefault(h, []).append((src_name, c, fpath))
                detailed_records.append({
                    'Source': src_name,
                    'Class': c,
                    'Filename': f,
                    'Hash': h,
                    'Valid': True,
                    'Duplicate': is_dup
                })
            except Exception:
                corrupt_count += 1
                detailed_records.append({
                    'Source': src_name,
                    'Class': c,
                    'Filename': f,
                    'Hash': 'ERROR',
                    'Valid': False,
                    'Duplicate': False
                })

        summary_data.append({
            'Source': src_name,
            'Class': c,
            'Total_Images': len(files),
            'Valid_Unique': valid_count,
            'Duplicates': dup_count,
            'Corrupt': corrupt_count
        })

df_summary = pd.DataFrame(summary_data)
print('\n--- VERIFICATION AUDIT TABLE ---')
print(df_summary.to_string(index=False))

# 2. Build Final_Combined Directory (1800 Images: 600 per class)
print('\n2. Populating Final_Combined directory...')
# Clean existing files in Final_Combined first
for c in CLASSES:
    dst_c = os.path.join(FINAL, c)
    for f in os.listdir(dst_c):
        try:
            os.remove(os.path.join(dst_c, f))
        except Exception:
            pass

    # Copy Internal (300)
    int_files = sorted([f for f in os.listdir(os.path.join(INTERNAL, c)) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])[:300]
    for idx, f in enumerate(int_files):
        src = os.path.join(INTERNAL, c, f)
        dst = os.path.join(dst_c, f'Grape_Int_{c}_{idx+1:04d}.jpg')
        shutil.copy2(src, dst)

    # Copy External (300)
    ext_files = sorted([f for f in os.listdir(os.path.join(EXTERNAL, c)) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])[:300]
    for idx, f in enumerate(ext_files):
        src = os.path.join(EXTERNAL, c, f)
        dst = os.path.join(dst_c, f'Grape_Ext_{c}_{idx+1:04d}.jpg')
        shutil.copy2(src, dst)

    total_combined = len(os.listdir(dst_c))
    print(f'  Final_Combined {c:<15}: {total_combined} images (300 Int + 300 Ext)')

# Save CSV reports
summary_csv = os.path.join(VERIF_DIR, 'Grape_Final_Summary.csv')
df_summary.to_csv(summary_csv, index=False)
detailed_csv = os.path.join(VERIF_DIR, 'Grape_Final_Detailed_Verification.csv')
pd.DataFrame(detailed_records).to_csv(detailed_csv, index=False)

print(f'\n✅ Audit Summary saved: {summary_csv}')
print(f'✅ Detailed Verification saved: {detailed_csv}')
print('=' * 80)
print('🎉 ALL 1,800 GRAPE IMAGES FULLY VERIFIED AND READY FOR PREPROCESSING!')
print('=' * 80)



🍇 GRAPE DATASET VERIFICATION & INTEGRITY AUDIT

--- VERIFICATION AUDIT TABLE ---
  Source       Class  Total_Images  Valid_Unique  Duplicates  Corrupt
INTERNAL     Healthy           300           300           0        0
INTERNAL   Black_Rot           300           300           0        0
INTERNAL Leaf_Blight           300           300           0        0
EXTERNAL     Healthy           300           300           0        0
EXTERNAL   Black_Rot           300           300           0        0
EXTERNAL Leaf_Blight           300           300           0        0

2. Populating Final_Combined directory...
  Final_Combined Healthy        : 600 images (300 Int + 300 Ext)
  Final_Combined Black_Rot      : 600 images (300 Int + 300 Ext)
  Final_Combined Leaf_Blight    : 600 images (300 Int + 300 Ext)

✅ Audit Summary saved: /content/drive/MyDrive/Plant Disease Detection (Computer Vision)/1st Raw_Data/Grape/Final_Verification/Grape_Final_Summary.csv
✅ Detailed Verification saved: /content/